<a href="https://colab.research.google.com/github/deburky/language-models/blob/main/colab-eval-all.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Model Evaluation: Multi-model comparison

Evaluates multiple models on knowledge Q&A and tool-calling tasks.  
Produces a side-by-side markdown report saved to `eval_report.md`.

**Models:**
- `deburky/gpt-oss-claude-code` — transformers (bf16, CUDA)
- `anthropic/claude-3-haiku` — OpenRouter API
- `Qwen/Qwen2.5-3B-Instruct` — transformers (bf16, CUDA)

Set `OPENROUTER_API_KEY` below to enable Haiku.

Authored by: [Denis Burakov](https://www.github.com/deburky)

## 1. Install dependencies

In [1]:
%pip install transformers torch datasets -q

In [2]:
from huggingface_hub import login
login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [3]:
# --- Set your OpenRouter API key here ---
import getpass
OPENROUTER_API_KEY = getpass.getpass()

··········


In [4]:
!nvidia-smi

Fri Apr  3 17:35:54 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX PRO 6000 Blac...    Off |   00000000:05:00.0 Off |                    0 |
| N/A   30C    P0             46W /  600W |       0MiB /  97887MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

## 2. Config

In [5]:
import contextlib
import glob as globlib
import json
import os
import pathlib
import re
import time
import urllib.request
from datetime import datetime

# Models to evaluate: label -> (model_id, backend)
# backend: "transformers" | "openrouter"
MODELS = {
    'gpt-oss-claude-code': ('deburky/gpt-oss-claude-code', 'transformers'),
    'claude-3-haiku': ('anthropic/claude-3-haiku', 'openrouter'),
    'Qwen2.5-3B': ('Qwen/Qwen2.5-3B-Instruct', 'transformers'),
}

KNOWLEDGE_PROMPTS = [
    'Who invented the method of Maximum Likelihood Estimation?',
    'Who is Alan Turing and what is he famous for?',
    'What is Weight of Evidence (WoE) in credit scoring and how is it calculated?',
    'Explain the difference between L1 and L2 regularization.',
    'What is the purpose of a chat template in large language models?',
]

TOOL_PROMPTS = [
    'List all Python files in the current directory.',
    'Read the first 20 lines of /etc/hostname.',
    'Search for the word root in /etc/hostname.',
]

TOOL_SYSTEM = """You are a coding assistant with access to the file system.

Available tools:
- read(path, offset, limit): Read a file. Use limit to cap lines returned.
- glob(pat): Find files by pattern
- grep(pat, path): Search for text in files

To use a tool, format it EXACTLY like this:
<tool_call>{"tool": "name", "args": {"key": "value"}}</tool_call>

When asked to read the first N lines, always pass limit: N. Reply concisely."""

MAX_TOKENS_KNOWLEDGE = 256
MAX_TOKENS_TOOL = 512
MAX_TOOL_STEPS = 5

print('Config ready.')
if not OPENROUTER_API_KEY:
    print('⚠ OPENROUTER_API_KEY not set — claude-3-haiku will be skipped')

Config ready.


## 3. Tool implementations and helpers

In [10]:
def tool_read(args):
    try:
        p = pathlib.Path(args['path'])
        lines = p.read_text(errors='replace').splitlines()
        offset = int(args.get('offset', 0))
        limit = int(args.get('limit', 20))
        return ''.join(f'{offset+i+1:4}| {l}\n' for i, l in enumerate(lines[offset:offset+limit]))
    except Exception as e:
        return f'error: {e}'


def tool_glob(args):
    try:
        pat = args.get('pat') or args.get('pattern', '*')
        files = globlib.glob(pat, recursive=True)
        return '\n'.join(sorted(files)) or 'none'
    except Exception as e:
        return f'error: {e}'


def tool_grep(args):
    try:
        pattern = re.compile(args.get('pat') or args.get('pattern', ''))
        p = pathlib.Path(args.get('path', '.'))
        candidates = [str(p)] if p.is_file() else globlib.glob(str(p / '**'), recursive=True)
        hits = []
        for fp in candidates:
            if not pathlib.Path(fp).is_file():
                continue
            with contextlib.suppress(Exception):
                for i, line in enumerate(open(fp, errors='replace'), 1):
                    if pattern.search(line):
                        hits.append(f'{fp}:{i}:{line.rstrip()}')
        return '\n'.join(hits[:40]) or 'none'
    except Exception as e:
        return f'error: {e}'


TOOLS = {'read': tool_read, 'glob': tool_glob, 'grep': tool_grep}


def run_tool(name, args):
    fn = TOOLS.get(name)
    return fn(args) if fn else f'error: unknown tool {name!r}'


def parse_tool_calls(text):
    calls = []
    # Native gpt-oss format: <read>{...}</read>
    for tag in ['read', 'glob', 'grep']:
        for m in re.finditer(rf'<{tag}>(.*?)</{tag}>', text, re.DOTALL):
            with contextlib.suppress(Exception):
                calls.append({'name': tag, 'args': json.loads(m.group(1))})
    # <tool_call>{"tool": "name", "args": {...}}</tool_call>
    for m in re.finditer(r'<tool_call>\s*(\{.*?\})\s*</tool_call>', text, re.DOTALL):
        with contextlib.suppress(Exception):
            d = json.loads(m.group(1))
            if d.get('tool') in TOOLS:
                calls.append({'name': d['tool'], 'args': d.get('args', {})})
    # Lenient: <tool_call>{...} without closing tag
    if not calls:
        for m in re.finditer(r'<tool_call>\s*(\{[^<]*)', text, re.DOTALL):
            with contextlib.suppress(Exception):
                s = m.group(1).strip()
                depth, last = 0, 0
                for i, c in enumerate(s):
                    if c == '{': depth += 1
                    elif c == '}':
                        depth -= 1
                        if depth == 0: last = i + 1
                d = json.loads(s[:last])
                if d.get('tool') in TOOLS:
                    calls.append({'name': d['tool'], 'args': d.get('args', {})})
    return calls


def strip_gpt_oss(text):
    if '<|channel|>final<|message|>' in text:
        text = text.split('<|channel|>final<|message|>')[-1]
    return re.sub(r'<\|[^>]+\|>', '', text).strip()


def parse_raw_to_message(raw):
    thinking, content = '', ''
    if '<|channel|>analysis<|message|>' in raw:
        part = raw.split('<|channel|>analysis<|message|>')[1]
        thinking = part.split('<|end|>')[0].strip()
    if '<|channel|>final<|message|>' in raw:
        part = raw.split('<|channel|>final<|message|>')[1]
        content = re.split(r'<\|end\|>|<\|return\|>', part)[0].strip()
    else:
        content = re.sub(r'<\|[^>]+\|>', '', raw).strip()
    msg = {'role': 'assistant', 'content': content}
    if thinking:
        msg['thinking'] = thinking
    return msg


print('Helpers ready.')

Helpers ready.


## 4. Inference backends

In [11]:
import gc
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer


def load_transformers_model(label, model_id):
    print(f'Loading {label} ({model_id})...')
    tok = AutoTokenizer.from_pretrained(model_id)
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    mdl = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=torch.bfloat16, device_map=device)
    print(f'  Loaded on {device}')
    return mdl, tok


def free_model(model):
    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def infer_transformers(model, tokenizer, messages, max_tokens):
    inputs = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True,
        return_tensors='pt', return_dict=True,
    ).to(model.device)
    t0 = time.perf_counter()
    with torch.no_grad():
        output_ids = model.generate(**inputs, max_new_tokens=max_tokens, temperature=0.3, do_sample=True)
    elapsed = time.perf_counter() - t0
    raw = tokenizer.decode(output_ids[0][inputs['input_ids'].shape[-1]:], skip_special_tokens=False)
    # Truncate at first complete tool call to prevent hallucination
    for tag in ['</tool_call>', '</read>', '</glob>', '</grep>']:
        idx = raw.find(tag)
        if idx != -1:
            raw = raw[:idx + len(tag)]
            break
    return strip_gpt_oss(raw), elapsed


def infer_openrouter(model_id, messages, max_tokens):
    payload = json.dumps({
        'model': model_id,
        'messages': messages,
        'max_tokens': max_tokens,
        'temperature': 0.3,
    }).encode()
    req = urllib.request.Request(
        'https://openrouter.ai/api/v1/chat/completions',
        data=payload,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {OPENROUTER_API_KEY}',
        },
    )
    t0 = time.perf_counter()
    with urllib.request.urlopen(req, timeout=60) as resp:
        data = json.loads(resp.read())
    elapsed = time.perf_counter() - t0
    return data['choices'][0]['message']['content'].strip(), elapsed


print('Backends ready.')

Backends ready.


## 5. Run evaluation

In [12]:
knowledge_results = [(p, {}) for p in KNOWLEDGE_PROMPTS]
tool_results = [(t, {}) for t in TOOL_PROMPTS]

for label, (model_id, backend) in MODELS.items():
    if backend == 'openrouter' and not OPENROUTER_API_KEY:
        print(f'Skipping {label} — OPENROUTER_API_KEY not set')
        continue

    print(f'\n{"=" * 60}')
    print(f'{label}  [{backend}]  {model_id}')
    print('=' * 60)

    model, tokenizer = (None, None)
    if backend == 'transformers':
        model, tokenizer = load_transformers_model(label, model_id)

    def infer(messages, max_tokens):
        if backend == 'openrouter':
            return infer_openrouter(model_id, messages, max_tokens)
        return infer_transformers(model, tokenizer, messages, max_tokens)

    print('[Knowledge Q&A]')
    for prompt, results in knowledge_results:
        print(f'Q: {prompt[:60]}...')
        try:
            messages = [
                {'role': 'system', 'content': 'You are a knowledgeable assistant. Answer concisely and accurately.'},
                {'role': 'user', 'content': prompt},
            ]
            answer, latency = infer(messages, MAX_TOKENS_KNOWLEDGE)
            results[label] = {'answer': answer, 'latency': latency}
            print(f'{latency:.2f}s — {answer[:80]}')
        except Exception as e:
            results[label] = {'answer': f'ERROR: {e}', 'latency': 0}
            print(f'ERROR: {e}')

    print('\n[Tool Calling]')
    for task, results in tool_results:
        print(f'  T: {task}')
        try:
            messages = [
                {'role': 'system', 'content': TOOL_SYSTEM},
                {'role': 'user', 'content': task},
            ]
            tool_log = []
            total_time = 0.0
            for _ in range(MAX_TOOL_STEPS):
                response, elapsed = infer(messages, MAX_TOKENS_TOOL)
                total_time += elapsed
                calls = parse_tool_calls(response)
                display = re.sub(
                    r'<tool_call>.*?</tool_call>|<(?:read|glob|grep)>.*?</(?:read|glob|grep)>', '',
                    response,
                    flags=re.DOTALL
                ).strip()
                tool_log.append({'response': display, 'tool_calls': []})
                if not calls:
                    break
                full_results = []
                for call in calls:
                    result = run_tool(call['name'], call['args'])
                    full_results.append(result)
                    tool_log[-1]['tool_calls'].append({
                        'tool': call['name'], 'args': call['args'],
                        'result': result[:800],
                    })
                messages.append({'role': 'assistant', 'content': response})
                messages.append({'role': 'user', 'content': '\n'.join(
                    f"Tool {c['name']} returned:\n{r}" for c, r in zip(calls, full_results)
                )})
            results[label] = {'log': tool_log, 'latency': total_time}
            print(f'{total_time:.2f}s — {len(tool_log)} step(s)')
        except Exception as e:
            results[label] = {'log': [], 'latency': 0}
            print(f'ERROR: {e}')

    if backend == 'transformers':
        free_model(model)

print('\nDone.')


gpt-oss-claude-code  [transformers]  deburky/gpt-oss-claude-code
Loading gpt-oss-claude-code (deburky/gpt-oss-claude-code)...


Loading weights:   0%|          | 0/411 [00:00<?, ?it/s]

  Loaded on cuda
[Knowledge Q&A]
Q: Who invented the method of Maximum Likelihood Estimation?...
2.31s — Maximum Likelihood Estimation (MLE) was formalized by Ronald Fisher in the early
Q: Who is Alan Turing and what is he famous for?...
2.49s — Alan Turing (1912–1954) was a British mathematician and computer scientist. He i
Q: What is Weight of Evidence (WoE) in credit scoring and how i...
2.20s — Weight of Evidence (WoE) is a transformation used in credit scoring to convert c
Q: Explain the difference between L1 and L2 regularization....
1.64s — L1 regularization adds a penalty equal to the absolute value of the coefficients
Q: What is the purpose of a chat template in large language mod...
0.98s — A chat template defines how user and system messages are formatted and tokenized

[Tool Calling]
  T: List all Python files in the current directory.
0.90s — 2 step(s)
  T: Read the first 20 lines of /etc/hostname.
2.32s — 2 step(s)
  T: Search for the word root in /etc/hostname.
0.98s — 2

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

  Loaded on cuda
[Knowledge Q&A]
Q: Who invented the method of Maximum Likelihood Estimation?...
0.70s — The method of Maximum Likelihood Estimation (MLE) was not invented by a single p
Q: Who is Alan Turing and what is he famous for?...
0.75s — Alan Turing is famous for his work during World War II, where he led the decrypt
Q: What is Weight of Evidence (WoE) in credit scoring and how i...
2.81s — Weight of Evidence (WoE) is a statistical technique used in credit scoring to me
Q: Explain the difference between L1 and L2 regularization....
1.77s — L1 (Lasso) and L2 (Ridge) regularization are techniques used to prevent overfitt
Q: What is the purpose of a chat template in large language mod...
0.69s — The purpose of a chat template in large language models is to provide a structur

[Tool Calling]
  T: List all Python files in the current directory.
0.18s — 1 step(s)
  T: Read the first 20 lines of /etc/hostname.
0.35s — 1 step(s)
  T: Search for the word root in /etc/hostname.
0.27s — 1

## 6. Build and save report

In [13]:
def fmt_tool_log(log):
    lines = []
    for step in log:
        if step['response']:
            lines.append(step['response'])
        for tc in step['tool_calls']:
            lines.append(f"\n**→ {tc['tool']}({json.dumps(tc['args'])})**")
            lines.append(f"```\n{tc['result']}\n```")
    return '\n'.join(lines).strip()


model_names = list(MODELS.keys())
col_sep = ' | '.join(model_names)
col_div = '|---|' + '---|' * len(model_names)
now = datetime.now().strftime('%Y-%m-%d %H:%M')

lines = [
    '# Model Evaluation Report',
    '',
    f'**Date:** {now}  ',
    '**Models:**',
]
lines.extend(f'- `{n}` → `{p}` ({b})' for n, (p, b) in MODELS.items())
lines += ['', '---', '', '## 1. Knowledge Q&A', '']

for prompt, results in knowledge_results:
    lines += [f'### Q: {prompt}', '']
    lines.extend((f'| | {col_sep} |', col_div))
    ans_row = '| **Answer** |'
    lat_row = '| **Latency** |'
    for m in model_names:
        r = results.get(m, {})
        ans_row += ' ' + r.get('answer', '*(no result)*').replace('\n', '<br>').replace('|', '\\|') + ' |'
        lat_row += f" {r.get('latency', 0):.2f}s |" if r else ' — |'
    lines += [ans_row, lat_row, '']

lines += ['---', '', '## 2. Tool Calling', '']
for task, results in tool_results:
    lines += [f'### Task: {task}', '']
    for m in model_names:
        r = results.get(m, {})
        lat = f"{r.get('latency', 0):.2f}s" if r else '—'
        log = fmt_tool_log(r.get('log', []))
        lines += [f'#### {m} — {lat}', '', log or '*(no output)*', '']

lines += ['---', '', '## 3. Latency Summary', '', f'| Task | {col_sep} |', col_div]
for prompt, results in knowledge_results:
    short = prompt[:60] + ('…' if len(prompt) > 60 else '')
    row = f'| {short} |'
    for m in model_names:
        r = results.get(m, {})
        row += f" {r.get('latency', 0):.2f}s |" if r else ' — |'
    lines.append(row)
for task, results in tool_results:
    short = task[:60] + ('…' if len(task) > 60 else '')
    row = f'| {short} (tool) |'
    for m in model_names:
        r = results.get(m, {})
        row += f" {r.get('latency', 0):.2f}s |" if r else ' — |'
    lines.append(row)

report = '\n'.join(lines)
ts = datetime.now().strftime('%Y%m%d_%H%M%S')
out_path = f'eval_{ts}.md'
with open(out_path, 'w') as f:
    f.write(report)
print(f'Report saved: {out_path}')
print(report[:800])

Report saved: eval_20260403_175032.md
# Model Evaluation Report

**Date:** 2026-04-03 17:50  
**Models:**
- `gpt-oss-claude-code` → `deburky/gpt-oss-claude-code` (transformers)
- `claude-3-haiku` → `anthropic/claude-3-haiku` (openrouter)
- `Qwen2.5-3B` → `Qwen/Qwen2.5-3B-Instruct` (transformers)

---

## 1. Knowledge Q&A

### Q: Who invented the method of Maximum Likelihood Estimation?

| | gpt-oss-claude-code | claude-3-haiku | Qwen2.5-3B |
|---|---|---|---|
| **Answer** | Maximum Likelihood Estimation (MLE) was formalized by Ronald Fisher in the early 20th century. He introduced it in his 1922 paper "On the Mathematical Foundations of Theoretical Statistics" and further developed it in his 1925 book *Statistical Methods for Research Workers* and 1930 book *Statistical Methods for Research Workers*. The method was later popul
